# Reading snapshot information (`snapinf_NNNN.txt`) and `error.txt`

| File | What it holds | Notebook |
|---|---|---|
| `galaxy_NNNN.pkl` or `.txt` (`halo_NNNN` for halos) | One row per object, properties as columns | `01_catalog.ipynb` |
| `galaxy_pid_NNNN.pkl` or `.txt` | Member particle IDs of each object (only if requested) | `02_members.ipynb` |
| `snapinf_NNNN.txt` | Snapshot metadata (`aexp`, aperture lists) | `03_snapinfo.ipynb` |
| `error.txt` | Requested snapshots that could not be produced (only if any) | `03_snapinfo.ipynb` |

Every snapshot in a download comes with a small text file of snapshot metadata. This notebook needs Python 3.7+, NumPy and `ndps_reader.py`, which sits one folder up.

The outputs shown come from a NewCluster galaxy download of snapshots 700–750.

In [1]:
import sys
sys.path.insert(0, '..')   # ndps_reader.py is one folder up; not needed if you pip-installed it

from pathlib import Path
import zipfile
import numpy as np
import ndps_reader as nr

print('ndps_reader', nr.__version__)

ndps_reader 0.1.0


## 1. Unzip the download

In [2]:
ZIP_FILE = Path('20261103_288872.zip')   # <- your download

def extract(zip_file):
    """Unzip a download once, into a folder named after it."""
    folder = zip_file.with_suffix('')
    if not folder.exists():
        with zipfile.ZipFile(zip_file) as archive:
            archive.extractall(folder)
    return folder

DATA_DIR = extract(ZIP_FILE)
files = sorted(p.name for p in DATA_DIR.iterdir())
print(len(files), 'files:', files[:4], '...', files[-2:])

53 files: ['error.txt', 'galaxy_0700.pkl', 'galaxy_0702.pkl', 'galaxy_0704.pkl'] ... ['snapinf_0748.txt', 'snapinf_0750.txt']


## 2. The file

Each line is `key = value`, and each value is written in JSON:

In [3]:
SNAP = 700
path = DATA_DIR / f'snapinf_{SNAP:04d}.txt'
print(path.read_text())

CONF_R = [1.0, 3.0]
GAS_R = [1.0, 2.0, 3.0]
GRAV_R = [5.0]
MAG_R = [-1.0, 0.1, 1.0]
Flux_List = ["NUV", "u", "g", "r", "i", "z"]
SFR_R = [-1.0, 0.1, 1.0, -1.0, 0.1, 1.0, -1.0, 0.1, 1.0, -1.0, 0.1, 1.0, -1.0, 0.1, 1.0]
SFR_T = [0.01, 0.01, 0.01, 0.05, 0.05, 0.05, 0.1, 0.1, 0.1, 0.5, 0.5, 0.5, 1.0, 1.0, 1.0]
aexp = [0.592947107301629]



`read_snapinfo` returns a dict:

In [4]:
info = nr.read_snapinfo(path)
for key, value in info.items():
    print(f'{key:10s} {type(value).__name__:5s} {value}')

CONF_R     list  [1.0, 3.0]
GAS_R      list  [1.0, 2.0, 3.0]
GRAV_R     list  [5.0]
MAG_R      list  [-1.0, 0.1, 1.0]
Flux_List  list  ['NUV', 'u', 'g', 'r', 'i', 'z']
SFR_R      list  [-1.0, 0.1, 1.0, -1.0, 0.1, 1.0, -1.0, 0.1, 1.0, -1.0, 0.1, 1.0, -1.0, 0.1, 1.0]
SFR_T      list  [0.01, 0.01, 0.01, 0.05, 0.05, 0.05, 0.1, 0.1, 0.1, 0.5, 0.5, 0.5, 1.0, 1.0, 1.0]
aexp       list  [0.592947107301629]


| Key | Contents |
|---|---|
| `aexp` | Scale factor of the snapshot, as a one-element list |
| `Flux_List` | Photometric bands: the `<band>` in `ABmag_<band>` |
| `MAG_R` | One value per position of `ABmag_<band>` |
| `GAS_R` | One value per position of `GasMass_<region>_<phase>` |
| `CONF_R` | One value per position of `ConFrac_M` and `ConFrac_N` |
| `GRAV_R` | The value used for `GravPressure` |
| `SFR_R`, `SFR_T` | One pair per position of `SFR` |

## 3. Redshift

In [5]:
aexp = info['aexp'][0]
print(f'aexp = {aexp:.5f}   z = {1 / aexp - 1:.4f}')

aexp = 0.59295   z = 0.6865


## 4. Labelling catalog vectors

Each list gives the meaning of the matching positions in a catalog vector (see `01_catalog.ipynb`):

In [6]:
cat = nr.read_catalog(DATA_DIR / f'galaxy_{SNAP:04d}.pkl')
g = cat[np.argmax(cat['Mass_tot'])]

for name, key in [('ABmag_r', 'MAG_R'), ('GasMass_tot_tot', 'GAS_R'), ('ConFrac_M', 'CONF_R')]:
    pairs = ', '.join(f'{r} -> {v:.4g}' for r, v in zip(info[key], g[name]))
    print(f'{name:16s} by {key}: {pairs}')

ABmag_r          by MAG_R: -1.0 -> -25.08, 0.1 -> -21.07, 1.0 -> -24.28
GasMass_tot_tot  by GAS_R: 1.0 -> 3.311e+10, 2.0 -> 3.596e+10, 3.0 -> 3.956e+10
ConFrac_M        by CONF_R: 1.0 -> 0, 3.0 -> 0


## 5. All snapshots in a download

In [7]:
print(' snap     aexp       z')
for p in sorted(DATA_DIR.glob('snapinf_*.txt')):
    a = nr.read_snapinfo(p)['aexp'][0]
    print(f'{int(p.stem[-4:]):5d} {a:8.5f} {1 / a - 1:7.4f}')

 snap     aexp       z
  700  0.59295  0.6865
  702  0.59386  0.6839
  704  0.59478  0.6813
  706  0.59570  0.6787
  708  0.59661  0.6761
  710  0.59752  0.6736
  712  0.59843  0.6710
  714  0.59934  0.6685
  716  0.60026  0.6659
  718  0.60118  0.6634
  720  0.60209  0.6609
  722  0.60301  0.6584
  724  0.60392  0.6559
  726  0.60483  0.6534
  728  0.60575  0.6509
  730  0.60666  0.6484
  732  0.60758  0.6459
  734  0.60849  0.6434
  736  0.60941  0.6409
  738  0.61031  0.6385
  740  0.61123  0.6360
  742  0.61215  0.6336
  744  0.61306  0.6312
  746  0.61397  0.6287
  748  0.61489  0.6263
  750  0.61581  0.6239


## 6. Without `ndps_reader`

The format is simple enough to read with the standard library alone:

In [8]:
import json

info2 = {}
for line in path.read_text().splitlines():
    key, _, value = line.partition(' = ')
    info2[key] = json.loads(value)
print(info2['aexp'], info2['Flux_List'])

[0.592947107301629] ['NUV', 'u', 'g', 'r', 'i', 'z']


## 7. `error.txt`

`error.txt` appears only when some requested snapshots could not be produced. It has one line per missing snapshot, and every other file in the download is complete. In this example the simulation only has even-numbered snapshots in this range, so the odd ones are listed.

In [9]:
import re

error_file = DATA_DIR / 'error.txt'
if error_file.exists():
    text = error_file.read_text()
    print(text.splitlines()[0])
    missing = [int(s) for s in re.findall(r'snapshot (\d+) is not available', text)]
    print(len(missing), 'missing snapshots:', missing)
else:
    print('No error.txt: every requested snapshot is included.')

Requested snapshot 701 is not available. (simulation=NewCluster, horg=g, file=snap_0701.hdf5)
25 missing snapshots: [701, 703, 705, 707, 709, 711, 713, 715, 717, 719, 721, 723, 725, 727, 729, 731, 733, 735, 737, 739, 741, 743, 745, 747, 749]
